In [ ]:
import pyspark
from delta import configure_spark_with_delta_pip
from delta import *
from commons import get_raw_data, filter_southern_region

builder = pyspark.sql.SparkSession.builder.appName("DeltaLab") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

spark = configure_spark_with_delta_pip(builder).getOrCreate()

: 

In [ ]:
df_vendas = get_raw_data(spark, "vendas.csv")
df_clientes = get_raw_data(spark, "clientes.csv")
df_sul = filter_southern_region(df_vendas.join(df_clientes, "id_cliente"))

In [ ]:
df_sul.write.format("delta").mode("overwrite").save("../data/delta/vendas_sul")

In [ ]:
# (Operações CRUD)
spark.read.format("delta").load("../data/delta/vendas_sul").createOrReplaceTempView("vendas_delta")
spark.sql("UPDATE vendas_delta SET valor = valor * 1.10 WHERE id_venda = 1")
spark.sql("DELETE FROM vendas_delta WHERE id_venda = 3")
spark.sql("SELECT * FROM vendas_delta").show()